# Independent Component Analysis (ICA)

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

ICA assumes the signal is a linear mixture of independent sources. We use MNE-Python to separate the 4 channels into independent components capturing brain activity and artifacts.

## Expected outputs

- Two plots: top shows original 4-channel EEG, bottom shows 4 ICA components
- Each component captures a different pattern of activity or artifacts
- The component with highest amplitude may be an artifact

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channels | 4 | P4, Cz, F8, T7 |
| n_components | 4 | Number of components |
| random_state | 97 | Random seed |
| max_iter | 800 | Max iterations |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply ICA

We create an MNE `Raw` object then apply `ICA` with 4 components.


In [ ]:
import mne

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

ica = mne.preprocessing.ICA(
    n_components=4, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)
components = ica.get_sources(raw).get_data()
print(f'ICA components shape: {components.shape}')


## 5. Interactive plot

**What to look for:**

- Each component captures a different pattern of activity
- The component with highest amplitude may be an artifact
- Use the zoom tool to inspect specific time ranges



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(eeg_data))
t_sec = np.arange(n_plot) / fs
colors = ['blue', 'orange', 'green', 'red']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original EEG Signal', 'ICA Independent Components'))
for i in range(4):
    offset = i * 200
    fig.add_trace(go.Scatter(x=t_sec, y=eeg_data[:n_plot, i] + offset,
                             name=ch_names[i], line=dict(color=colors[i], width=0.5)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=t_sec, y=components[i, :n_plot] * 1e6 + offset,
                             name=f'IC{i}', line=dict(color=colors[i], width=0.5)),
                  row=2, col=1)
fig.update_layout(height=700, title_text='ICA Decomposition - Artifact Separation',
                  xaxis2_title='Time (s)', showlegend=True)
fig.show()


## What did we learn?

- ICA separates the signal into independent sources without prior knowledge
- It assumes source independence and non-Gaussianity
- It works better with a larger number of channels
- The highest-amplitude component is often an artifact

